# [stand-in] CubeLang emitter — SFT with Unsloth, export to GGUF

**What this is.** A small open model fine-tuned to emit CubeLang programs from a question, so the
serve stack (VM verification, worlds, harvest, the learning gate as a live loop) can be built while
the 2B CubbyLLM trunk trains. **It is a stand-in**: nothing it measures is a CubbyLLM result — no
hybrid backbone, no θ=f(c), no bounded state, no episodic store. Tag every number it produces
`[stand-in]`; it lives behind the trunk interface in `standin/`, outside `cubbyllm/`, and is swapped
out the day the 2B checkpoint exists.

**Data.** `standin/data/out/emitter_sft.jsonl`, built by `standin/data/build_emitter_sft.py` and
re-verified program-by-program through the real Rust cubelang VM: GSM8K-*train*-derived arithmetic
programs (GSM8K test excluded — it is H-G4's eval), role-binding programs (capped), the four
reasoning kernels, and our 517 verified multi-hop chain programs (`ISolve`/`recover` dialect — the
ones the oracle actually needs). Upload the jsonl + manifest to `Drive/cubbyllm/standin/` first.

**Model.** `LiquidAI/LFM2.5-2.6B` (2026-08 Hub check: best structured-output profile among ≤4B —
IFStruct 85.5 / Multi-IF 80.1 / BFCL v4 56.9 self-reported; LFM Open License, fine for a non-commercial
prototype). Fallback: `ibm-granite/granite-4.2-3b` (Apache-2.0, stronger raw code) once Unsloth artifacts land.

**Identity + hormones.** The set also carries ~300 identity turns (Cubby / Grillcheese Research Lab / not AGI /
how it answers) under their OWN system prompt, which includes a sampled **hormonal state** (dopamine,
serotonin, cortisol, oxytocin, noradrenaline — the cubbyverse demo's `neurochemistry.py` set and bands)
and the register it implies. The host injects that block at serve time, so "everything is modulated by
hormones" is true by construction; tone/caution change, facts never do; the emitter turns carry no state.
Identity turns are **bilingual (EN/FR)**: a French question gets a French answer, and the don't-know line
is verbatim in the question's language. Every record names its `system` prompt — the formatter below uses it.

**Eval split.** Colab does text exact-match on the held-out `val` split (cheap, format-level).
The real read — *does the emitted program execute on the VM and hit gold* — runs **locally** after
this notebook exports the GGUF (`standin/eval_emitter_vm.py`, llama.cpp Vulkan + `cubelang.exe`).

In [ ]:
# --- setup (run once per session) ---
import os, json, time, random, re
!pip -q install unsloth trl datasets
import sys
if not os.path.exists('/content/CubbyLLM'):
    !git clone -q https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM
else:
    !cd /content/CubbyLLM && git pull -q --ff-only
sys.path.insert(0, '/content/CubbyLLM/standin/data'); sys.path.insert(0, '/content/CubbyLLM')
from identity import identity_ok, load_facts, EMITTER_SYSTEM   # the identity check + facts
FACTS = load_facts()
from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cubbyllm/standin'
DATA = f'{DRIVE}/emitter_sft.jsonl'
MANIFEST = f'{DRIVE}/emitter_sft.manifest.json'
OUT = f'{DRIVE}/emitter_lfm25_2p6b'          # adapter + merged + GGUF land here
MODEL = 'LiquidAI/LFM2.5-2.6B'
MAX_SEQ = 2048
!nvidia-smi --query-gpu=name,memory.total --format=csv
for f in (DATA, MANIFEST):
    print(('ok      ' if os.path.exists(f) else 'MISSING ') + f)
m = json.load(open(MANIFEST)); print('manifest:', m['by_task'], '| kept', m['n_records_kept'], '| built', m['built'][:19], '| git', m['git_rev'][:8])

In [ ]:
# --- data: chat-format the records; train/val from the builder's deterministic split ---
SYSTEM = EMITTER_SYSTEM   # emitter turns; identity turns carry their own r['system'] with the hormonal state
recs = [json.loads(l) for l in open(DATA, encoding='utf-8')]
recs = [r for r in recs if r.get('vm_ok') in (True, None) and r.get('gold_match') is not False]
train = [r for r in recs if r['split'] == 'train']; val = [r for r in recs if r['split'] == 'val']
from collections import Counter
print('train', len(train), Counter(r['task'] for r in train)); print('val  ', len(val), Counter(r['task'] for r in val))

def to_messages(r):
    return [{'role': 'system', 'content': r.get('system') or SYSTEM},
            {'role': 'user', 'content': r['prompt']},
            {'role': 'assistant', 'content': r['program'].strip() + '\n'}]

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(MODEL, max_seq_length=MAX_SEQ, load_in_4bit=False, dtype=None)

def fmt(r):
    return {'text': tokenizer.apply_chat_template(to_messages(r), tokenize=False, add_generation_prompt=False)}
from datasets import Dataset
random.Random(0).shuffle(train)
ds_train = Dataset.from_list([fmt(r) for r in train])
lens = [len(tokenizer(x['text']).input_ids) for x in ds_train.select(range(min(500, len(ds_train))))]
print('token lengths (sample of 500): max', max(lens), 'p95', sorted(lens)[int(0.95*len(lens))], '-> MAX_SEQ', MAX_SEQ)
print(ds_train[0]['text'][:900])

In [ ]:
# --- LoRA + SFT (one epoch; ~4-5M tokens; minutes on an A100) ---
from trl import SFTTrainer, SFTConfig
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0.0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj', 'o_proj', 'in_proj', 'w1', 'w2', 'w3', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=0)
cfg = SFTConfig(output_dir='/content/emitter_ckpt', per_device_train_batch_size=8, gradient_accumulation_steps=4,
                num_train_epochs=1, learning_rate=2e-4, lr_scheduler_type='cosine', warmup_steps=20,
                logging_steps=10, save_strategy='no', bf16=True, max_seq_length=MAX_SEQ, dataset_text_field='text',
                packing=False, report_to='none', seed=0)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds_train, args=cfg)
t0 = time.time(); stats = trainer.train(); print(f'trained in {(time.time()-t0)/60:.1f} min; final loss', stats.training_loss)
os.makedirs(OUT, exist_ok=True); model.save_pretrained(f'{OUT}/adapter'); tokenizer.save_pretrained(f'{OUT}/adapter')
print('adapter ->', f'{OUT}/adapter')

In [ ]:
# --- format-level eval on the held-out val split: greedy, exact match after whitespace normalisation ---
# This is NOT the verified read (the Rust VM is not on Colab) -- see standin/eval_emitter_vm.py for that.
FastLanguageModel.for_inference(model)
def norm(s): return re.sub(r'\s+', ' ', re.sub(r'#.*', '', s)).strip()   # drop comments, collapse whitespace
def emit(prompt, max_new=768, system=None):
    ids = tokenizer.apply_chat_template([{'role': 'system', 'content': system or SYSTEM}, {'role': 'user', 'content': prompt}],
                                        tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(ids, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
N_EVAL = 200
rng = random.Random(1); sample = rng.sample(val, min(N_EVAL, len(val)))
hits = Counter(); tot = Counter(); outputs = []
t0 = time.time()
for i, r in enumerate(sample):
    if r['task'] == 'identity':
        gen = emit(r['prompt'], max_new=200, system=r.get('system')); ok = identity_ok(r.get('subtype', ''), gen, FACTS, r.get('lang', 'en'))
    else:
        gen = emit(r['prompt']); ok = norm(gen) == norm(r['program'])
    tot[r['task']] += 1; hits[r['task']] += int(ok)
    outputs.append({'id': r['id'], 'task': r['task'], 'subtype': r.get('subtype', ''), 'prompt': r['prompt'],
                    'reference': r['program'], 'generated': gen, 'exact_match': ok, 'gold': r.get('gold'),
                    'system': r.get('system'), 'lang': r.get('lang')})
    if (i+1) % 50 == 0: print(f'  {i+1}/{len(sample)} ({time.time()-t0:.0f}s)')
print('[stand-in] val by task (identity = identity_ok, others = text exact-match):', {t: f'{hits[t]}/{tot[t]}' for t in tot}, '| overall', sum(hits.values())/sum(tot.values()))
json.dump({'model': MODEL, 'n': len(sample), 'exact_match_by_task': {t: hits[t]/tot[t] for t in tot},
           'outputs': outputs, 'manifest_output_sha256': m['output_sha256']}, open(f'{OUT}/val_generations.json', 'w'), indent=1)
print('generations ->', f'{OUT}/val_generations.json  (the local VM eval reads this file too)')

In [ ]:
# --- export: merged fp16 + GGUF (q8_0 for the VM eval, q4_k_m for the 12 GB Vulkan box) ---
model.save_pretrained_merged(f'{OUT}/merged', tokenizer, save_method='merged_16bit')
model.save_pretrained_gguf(f'{OUT}/gguf', tokenizer, quantization_method=['q8_0', 'q4_k_m'])
!ls -la {OUT}/gguf
print('done -> download the GGUF, then locally: python standin/eval_emitter_vm.py --gguf <file> --val-generations', f'{OUT}/val_generations.json')

### How to read

- **Exact match is a format read, not a capability read.** Arithmetic programs have one canonical
  decomposition per question in the data, so EM is meaningful there; role-binding EM mostly tests
  whether the model picked the same ACTION/AGENT split; chain programs are deterministic given the
  retrieved facts, so EM is fair. Anything the model emits that *differs but executes to gold* only shows
  up in the local VM eval — expect VM-verified accuracy ≥ EM.
- **Report per task.** The chain task is the one the oracle needs; a high overall EM carried by
  role-binding is not a result.
- **Identity is scored by `identity_ok`**, not exact match: name/builder present, never affirms AGI or
  feelings, never claims to be another model, and the affect turns must speak in state/register language
  that matches the injected hormonal block. Try a few live prompts with different `state` blocks to see the
  register shift (cautious vs curious vs warm).
- **Everything here is `[stand-in]`.** It goes in `standin/README.md` and the TODO, never in
  `CUBBYLLM_HYPOTHESES.md` except as a pointer.